In [ ]:
!pip install -q transformers datasets accelerate torch

In [20]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
)

set_seed(42)  # set seed for reproducibility

print('GPU available:', torch.cuda.is_available())

GPU available: True


In [21]:
import json

# Load training data
with open('train.json', 'r', encoding='utf-8') as f:
    train_examples = json.load(f)

# Load validation data
with open('validation.json', 'r', encoding='utf-8') as f:
    validation_examples = json.load(f)

# Check token and tag counts
for example in train_examples + validation_examples:
    assert len(example['tokens']) == len(example['tags'])

print('Training examples:', len(train_examples))
print('Validation examples:', len(validation_examples))

Training examples: 166
Validation examples: 56


In [22]:
# @title create label-to-id and id-to-label mapping

label_list = [
    'O',
    'B-CHEM_ENT',
    'I-CHEM_ENT',
    'B-MAT_ENT_STRUCT',
    'I-MAT_ENT_STRUCT',
    'B-MAT_ENT_UNSTRUCT',
    'I-MAT_ENT_UNSTRUCT',
    'B-PROPERTY',
    'I-PROPERTY',
    'B-END_USE',
    'I-END_USE',
    'B-PROCESS',
    'I-PROCESS',
    'B-EQUIPMENT',
    'I-EQUIPMENT',
    'B-MEASUREMENT',
    'I-MEASUREMENT',
    'B-ABBREVIATION',
    'I-ABBREVIATION',
]

# map label names to numeric ids
label2id = {}

for index, label in enumerate(label_list):
    label2id[label] = index

# map numeric ids back to label names
id2label = {}

for label, index in label2id.items():
    id2label[index] = label

print(label2id)
print(id2label)


{'O': 0, 'B-CHEM_ENT': 1, 'I-CHEM_ENT': 2, 'B-MAT_ENT_STRUCT': 3, 'I-MAT_ENT_STRUCT': 4, 'B-MAT_ENT_UNSTRUCT': 5, 'I-MAT_ENT_UNSTRUCT': 6, 'B-PROPERTY': 7, 'I-PROPERTY': 8, 'B-END_USE': 9, 'I-END_USE': 10, 'B-PROCESS': 11, 'I-PROCESS': 12, 'B-EQUIPMENT': 13, 'I-EQUIPMENT': 14, 'B-MEASUREMENT': 15, 'I-MEASUREMENT': 16, 'B-ABBREVIATION': 17, 'I-ABBREVIATION': 18}
{0: 'O', 1: 'B-CHEM_ENT', 2: 'I-CHEM_ENT', 3: 'B-MAT_ENT_STRUCT', 4: 'I-MAT_ENT_STRUCT', 5: 'B-MAT_ENT_UNSTRUCT', 6: 'I-MAT_ENT_UNSTRUCT', 7: 'B-PROPERTY', 8: 'I-PROPERTY', 9: 'B-END_USE', 10: 'I-END_USE', 11: 'B-PROCESS', 12: 'I-PROCESS', 13: 'B-EQUIPMENT', 14: 'I-EQUIPMENT', 15: 'B-MEASUREMENT', 16: 'I-MEASUREMENT', 17: 'B-ABBREVIATION', 18: 'I-ABBREVIATION'}


In [23]:
# define a function to convert tag names to numeric ids
# and create a hugging face dataset
def make_dataset(examples):
    data = []

    for example in examples:
        tag_ids = []

        # replace each tag name with its numeric id
        for tag in example['tags']:
            tag_id = label2id[tag]
            tag_ids.append(tag_id)

        item = {
            'tokens': example['tokens'],
            'ner_tags': tag_ids,
        }

        data.append(item)

    # convert the list of dictionaries into a hugging face dataset
    dataset = Dataset.from_list(data)

    return dataset

In [24]:
# create hugging face datasets for training and validation
train_dataset = make_dataset(train_examples)
validation_dataset = make_dataset(validation_examples)

# print the dataset summary
print(train_dataset)

# print the actual data in the first row
print(train_dataset[0])

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 166
})
{'tokens': ['The', 'cryo', '-', 'fractured', 'surface', 'of', 'PP', 'samples', 'was', 'coated', 'with', 'gold', 'vapor', 'and', 'observed', 'under', 'SEM', '(', 'JSM', '-', '7400F', ',', 'JEOL', ',', 'Japan', ')', 'to', 'evaluate', 'the', 'distribution', 'of', 'CNC', 'in', 'the', 'PP', 'matrix', '.'], 'ner_tags': [0, 11, 12, 12, 0, 0, 5, 6, 0, 11, 0, 3, 4, 0, 0, 0, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 5, 6, 0]}


In [25]:
# load the tokenizer associated with this BERT checkpoint
checkpoint = 'google-bert/bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint,
    use_fast=True,
)

In [26]:
# define a function to tokenize sentences and align labels with tokens
def tokenize_and_align_labels(examples):

    # with batched=True, examples contains lists grouped by column:
    # {
    #     'tokens': [sentence_1_tokens, sentence_2_tokens, ...],
    #     'ner_tags': [sentence_1_labels, sentence_2_labels, ...],
    # }

    # tokenize all sentences in this batch
    tokenized = tokenizer(
        examples['tokens'],
        is_split_into_words=True,
        truncation=True,
    )

    aligned_labels = []

    # process one sentence's label list at a time
    for index, word_labels in enumerate(examples['ner_tags']):

        # map each BERT token to its original word position
        # special tokens have word_id=None
        word_ids = tokenized.word_ids(batch_index=index)

        previous_word_id = None
        token_labels = []

        for word_id in word_ids:

            if word_id is None:
                token_labels.append(-100)

            elif word_id != previous_word_id:
                label = word_labels[word_id]
                token_labels.append(label)

            else:
                token_labels.append(-100)

            previous_word_id = word_id

        # store the aligned labels for this sentence
        aligned_labels.append(token_labels)

    # the model expects the target labels under the key 'labels'
    tokenized['labels'] = aligned_labels

    return tokenized

In [27]:
# create new tokenized datasets
tokenized_train = train_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_validation = validation_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=validation_dataset.column_names,
)

print(tokenized_train[0])

Map:   0%|          | 0/166 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

{'input_ids': [101, 1109, 5354, 1186, 118, 25555, 2473, 1104, 27660, 8025, 1108, 16055, 1114, 2284, 23548, 1105, 4379, 1223, 12342, 2107, 113, 147, 16450, 118, 5692, 7629, 2271, 117, 147, 2036, 13901, 117, 1999, 114, 1106, 17459, 1103, 3735, 1104, 140, 15517, 1107, 1103, 27660, 8952, 119, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 0, 11, -100, 12, 12, 0, 0, 5, 6, 0, 11, 0, 3, 4, 0, 0, 0, 13, -100, 0, 0, -100, 0, 0, -100, -100, 0, 0, -100, -100, 0, 0, 0, 0, 0, 0, 0, 0, 5, -100, 0, 0, 5, 6, 0, -100]}


In [29]:
# inspect the BERT tokens and their aligned labels for one sentence
sample = tokenized_train[0]

tokens = tokenizer.convert_ids_to_tokens(sample['input_ids'])

for token, label_id in zip(tokens, sample['labels']):

    if label_id == -100:
        label = 'IGNORED'
    else:
        label = id2label[label_id]

    print(token, '->', label)

[CLS] -> IGNORED
The -> O
cry -> B-PROCESS
##o -> IGNORED
- -> I-PROCESS
fractured -> I-PROCESS
surface -> O
of -> O
PP -> B-MAT_ENT_UNSTRUCT
samples -> I-MAT_ENT_UNSTRUCT
was -> O
coated -> B-PROCESS
with -> O
gold -> B-MAT_ENT_STRUCT
vapor -> I-MAT_ENT_STRUCT
and -> O
observed -> O
under -> O
SE -> B-EQUIPMENT
##M -> IGNORED
( -> O
J -> O
##SM -> IGNORED
- -> O
74 -> O
##00 -> IGNORED
##F -> IGNORED
, -> O
J -> O
##E -> IGNORED
##OL -> IGNORED
, -> O
Japan -> O
) -> O
to -> O
evaluate -> O
the -> O
distribution -> O
of -> O
C -> B-MAT_ENT_UNSTRUCT
##NC -> IGNORED
in -> O
the -> O
PP -> B-MAT_ENT_UNSTRUCT
matrix -> I-MAT_ENT_UNSTRUCT
. -> O
[SEP] -> IGNORED


In [30]:
# load pretrained BERT with a token classification layer for NER
# the classification layer starts with newly initialized weights
model = AutoModelForTokenClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params 

In [31]:
# pad sentences to the longest sequence in each training batch
# padding positions receive attention_mask=0 and labels=-100
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
)

In [32]:
training_args = TrainingArguments(
    output_dir='./models',
    learning_rate=1e-5,             # Size of weight updates.
    per_device_train_batch_size=2,  # Training sentences per batch per device.
    per_device_eval_batch_size=2,   # Validation sentences per batch per device.
    num_train_epochs=10,             # Ten passes through this small training set.
    eval_strategy='epoch',          # Evaluate after each epoch.
    save_strategy='no',             # Save manually after training.
    logging_steps=10,
    report_to='none',               # Disable external logging services.
    seed=42,
)


In [33]:
# Connect the model, datasets, tokenizer, and training settings.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator,
)


In [34]:
print(checkpoint)

google-bert/bert-base-cased


In [35]:
# Start fine-tuning: update BERT and the NER classification layer.
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.928205,0.911488
2,0.801092,0.690382
3,0.554953,0.601296
4,0.563705,0.558991
5,0.425689,0.529159
6,0.452201,0.525037
7,0.386966,0.514483
8,0.244403,0.506988
9,0.313240,0.511252
10,0.394024,0.514833


TrainOutput(global_step=830, training_loss=0.5637115929500166, metrics={'train_runtime': 63.9294, 'train_samples_per_second': 25.966, 'train_steps_per_second': 12.983, 'total_flos': 256861522288512.0, 'train_loss': 0.5637115929500166, 'epoch': 10.0})

In [36]:
!pip install seqeval -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [37]:
import numpy as np
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

predictions, labels, metrics = trainer.predict(tokenized_validation)

predicted_ids = np.argmax(predictions, axis=2)

true_predictions = []
true_labels = []

for prediction, label in zip(predicted_ids, labels):

    predicted_tags = []
    actual_tags = []

    for pred_id, label_id in zip(prediction, label):

        if label_id != -100:
            predicted_tags.append(label_list[pred_id])
            actual_tags.append(label_list[label_id])

    true_predictions.append(predicted_tags)
    true_labels.append(actual_tags)

print('Precision:', precision_score(true_labels, true_predictions))
print('Recall:', recall_score(true_labels, true_predictions))
print('F1-score:', f1_score(true_labels, true_predictions))

print('\nPer-label performance:')
print(classification_report(true_labels, true_predictions, digits=4))

Precision: 0.4490223463687151
Recall: 0.5453774385072095
F1-score: 0.4925315970892378

Per-label performance:
                  precision    recall  f1-score   support

    ABBREVIATION     0.3600    0.5902    0.4472        61
        CHEM_ENT     0.4459    0.6034    0.5128       116
         END_USE     0.3000    0.1667    0.2143        18
       EQUIPMENT     0.0800    0.0323    0.0460        62
  MAT_ENT_STRUCT     0.0000    0.0000    0.0000        33
MAT_ENT_UNSTRUCT     0.4872    0.6382    0.5526       387
     MEASUREMENT     0.7066    0.7564    0.7307       156
         PROCESS     0.3725    0.4903    0.4234       155
        PROPERTY     0.3473    0.4764    0.4018       191

       micro avg     0.4490    0.5454    0.4925      1179
       macro avg     0.3444    0.4171    0.3699      1179
    weighted avg     0.4299    0.5454    0.4781      1179



/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [38]:
# Calculate validation loss without updating model weights.
# This does not calculate NER precision, recall, or F1.
evaluation_results = trainer.evaluate()

print(evaluation_results)

Training Loss,Validation Loss,Epoch
0.394024,0.514833,10


{'eval_loss': 0.514833390712738}


In [ ]:
# Create an inference pipeline using the fine-tuned model.
# Use the first subtoken's prediction to choose each word's label.
ner = pipeline(
    task='token-classification',
    model=trainer.model,
    tokenizer=tokenizer,
    aggregation_strategy='first',
    device=0 if torch.cuda.is_available() else -1,
)

In [ ]:
text = 'bacterial cellulose was mixed with ethanol.'

# Predict entities in the new sentence.
# This tiny training dataset may produce incorrect predictions.
results = ner(text)

for entity in results:
    print({
        'text': text[entity['start']:entity['end']],
        'label': entity['entity_group'],
        'score': round(entity['score'], 3),
    })

{'text': 'bacterial cellulose', 'label': 'MATERIAL', 'score': np.float32(0.932)}
{'text': 'ethanol', 'label': 'CHEMICAL', 'score': np.float32(0.969)}


In [ ]:
# Save both the fine-tuned model and its tokenizer.
# Files saved here are temporary unless downloaded or copied elsewhere.
trainer.save_model('./my_bert_ner')
tokenizer.save_pretrained('./my_bert_ner')

{'text': 'bacterial cellulose', 'label': 'MATERIAL', 'score': np.float32(0.803)}
{'text': 'ethanol', 'label': 'CHEMICAL', 'score': np.float32(0.962)}


In [ ]:
# Reload the saved model and tokenizer without training again.
# Use CPU to avoid loading another copy onto the GPU.
reloaded_ner = pipeline(
    task='token-classification',
    model='./my_bert_ner',
    tokenizer='./my_bert_ner',
    aggregation_strategy='first',
    device=-1,
)

results = reloaded_ner(text)

for entity in results:
    print({
        'text': text[entity['start']:entity['end']],
        'label': entity['entity_group'],
        'score': round(entity['score'], 3),
    })

In [ ]:
# Optional: download the saved model and tokenizer from Colab as a ZIP file.
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    'my_bert_ner',
    'zip',
    root_dir='./my_bert_ner',
)

files.download(archive_path)